# 📌 Importação de bibliotecas

In [1]:
import numpy as np
import pandas as pd

In [2]:
from typing import Union

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [5]:
from scipy.stats import f_oneway
from scipy.stats import levene
from scipy import stats
import pingouin as pg

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, KFold
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from yellowbrick.model_selection import FeatureImportances

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error, mean_absolute_percentage_error
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import average_precision_score
from sklearn.metrics import classification_report
from yellowbrick.classifier import ClassificationReport
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc, precision_recall_curve
)

In [7]:
import copy

In [8]:
import warnings

## Importação de pacotes locais

In [9]:
import src.local_tools as lt
import src.telecomx_analysis as ta
import src.telecomx_machine_learning as tml

## Configurações do ambiente

In [10]:
pd.set_option('display.max_columns', None)

In [11]:
warnings.simplefilter(action='ignore', category=FutureWarning)

## Constantes

In [12]:
NUM_SEMENTE_ALEATORIA = 42
TAMANHO_TESTE = 0.3
MAXIMO_ITERACAO = 1000

In [13]:
LIST_SCORING = ['accuracy','recall', 'precision', 'f1']

# 📌 Extração de dados

In [14]:
df_dados = pd.read_csv('./dados/dados_tratados.csv')

In [15]:
df_dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customerID                    7043 non-null   object 
 1   Churn                         7043 non-null   int64  
 2   customer_gender               7043 non-null   object 
 3   customer_SeniorCitizen        7043 non-null   int64  
 4   customer_Partner              7043 non-null   int64  
 5   customer_Dependents           7043 non-null   int64  
 6   customer_tenure               7043 non-null   int64  
 7   phone_PhoneService            7043 non-null   int64  
 8   phone_MultipleLines           7043 non-null   int64  
 9   internet_InternetService      7043 non-null   int64  
 10  internet_OnlineSecurity       7043 non-null   int64  
 11  internet_OnlineBackup         7043 non-null   int64  
 12  internet_DeviceProtection     7043 non-null   int64  
 13  int

In [16]:
df_dados.nunique()

customerID                      7043
Churn                              2
customer_gender                    2
customer_SeniorCitizen             2
customer_Partner                   2
customer_Dependents                2
customer_tenure                   72
phone_PhoneService                 2
phone_MultipleLines                2
internet_InternetService           2
internet_OnlineSecurity            2
internet_OnlineBackup              2
internet_DeviceProtection          2
internet_TechSupport               2
internet_StreamingTV               2
internet_StreamingMovies           2
account_Contract                   3
account_PaperlessBilling           2
account_PaymentMethod              4
account_Charges_Monthly         1585
account_Charges_Total           6534
internet_Service_Description       3
customer_tenure_bins               6
account_Charges_Monthly_bins       6
account_Charges_Total_bins        13
account_Contract_Monthly           2
additional_InternetService         7
o

Verificar dados duplicados

In [17]:
df_dados.duplicated().sum()

0

# 📌 Tratamento de dados

In [18]:
df_ohe = tml.df_final_modelo_v1(df_dados)

# 📌 Seleção e validação dos modelos de treinamento

## Tabela de verificação de padronização dos dados

| Modelo                 | Precisa padronizar? | Tipo de padronização (se necessário)   |
| ---------------------- | ------------------- | -------------------------------------- |
| DecisionTreeClassifier | ❌ Não               | —                                      |
| LogisticRegression     | ✅ Sim               | `StandardScaler` (média 0, desvio 1)   |
| RandomForest           | ❌ Não               | —                                      |
| XGBoost                | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| LightGBM               | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| CatBoost               | ❌ Não               | —                                      |


## Conceitos sobre as métricas de um modelo

Aqui está o quadro comparativo para churn (evasão de clientes), considerando que a classe positiva é “cliente vai sair”:

| **Métrica**                | **O que mede**                                                                     | **Quando valor é alto**                                        | **Risco quando valor é baixo**                                   | **Custo de erro associado**                                                |
| -------------------------- | ---------------------------------------------------------------------------------- | -------------------------------------------------------------- | ---------------------------------------------------------------- | -------------------------------------------------------------------------- |
| **Precisão (Precision)**   | Entre todos que o modelo previu como “vai sair”, qual porcentagem realmente saiu   | Você gasta retenção apenas em quem de fato sairia              | Gastar recursos em retenção de clientes que iam ficar (FP alto)  | **Custo financeiro** com campanhas desnecessárias (descontos, bônus, etc.) |
| **Recall (Sensibilidade)** | Entre todos que realmente saíram, qual porcentagem o modelo previu como “vai sair” | Você identifica a maior parte dos clientes que iam sair        | Deixar escapar clientes que saem (FN alto)                       | **Perda de receita** e potencial perda de market share                     |
| **F1-Score**               | Média harmônica de precisão e recall                                               | Equilíbrio entre acertar quem vai sair e evitar falsos alarmes | Ou alto custo de retenção inútil ou perda de clientes — ou ambos | **Equilíbrio financeiro e estratégico** — custo total menor                |
| **FP (Falso Positivo)**    | Previu saída, mas o cliente ficaria                                                | —                                                              | Gastar para reter quem não ia sair                               | Desperdício de budget de retenção                                          |
| **FN (Falso Negativo)**    | Previu permanência, mas o cliente saiu                                             | —                                                              | Não agir para reter quem realmente sairia                        | Perda direta de receita + possível impacto na reputação                    |
    

📌 Resumo visual da prioridade

    Se retenção for muito cara → priorizar alta precisão.

    Se perder clientes for muito prejudicial → priorizar alto recall.

    Se quer balancear ambos → otimizar F1-Score.

## Split base de dados em train e test

In [19]:
X = df_ohe.drop(columns=['Churn'])

In [20]:
y = df_ohe.Churn

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = NUM_SEMENTE_ALEATORIA, test_size=TAMANHO_TESTE, stratify=y)

## Treinamento do Modelo - LightGBM

In [22]:
import lightgbm as lgb

In [23]:
# Identificar variáveis categóricas
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Codificar variáveis categóricas com LabelEncoder
# (LightGBM aceita categorias inteiras, mas não strings)
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le  # guardar encoders caso queira reverter depois

In [24]:
# Criar modelo LightGBM
model_lgb = lgb.LGBMClassifier(
    boosting_type='gbdt',
    num_leaves=31,
    max_depth=-1,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=NUM_SEMENTE_ALEATORIA
)

In [25]:
# Treinar
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='auc',
    categorical_feature=categorical_cols,
    early_stopping_rounds=50,
    verbose=False
)

C:\ProgramData\anaconda3\Lib\site-packages\lightgbm\sklearn.py:726: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
C:\ProgramData\anaconda3\Lib\site-packages\lightgbm\sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "
C:\ProgramData\anaconda3\Lib\site-packages\lightgbm\basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
C:\ProgramData\anaconda3\Lib\site-packages\lightgbm\basic.py:2068: UserWarning: categorical_feature in Dataset is overridd

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
               random_state=42, subsample=0.8)

In [26]:
# Previsões
y_pred_lgb = model_lgb.predict(X_test)
y_pred_proba_lgb = model_lgb.predict_proba(X_test)

### Relatório de métricas

In [27]:
tml.avaliar_modelo(y_test, y_pred_lgb, y_pred_proba_lgb, False)

Métricas:
Acurácia: 0.7733
Precisão: 0.7847
Recall: 0.2014
F1-Score: 0.3206
ROC AUC: 0.8296

Relatório de classificação:

              precision    recall  f1-score   support

           0       0.77      0.98      0.86      1552
           1       0.78      0.20      0.32       561

    accuracy                           0.77      2113
   macro avg       0.78      0.59      0.59      2113
weighted avg       0.78      0.77      0.72      2113


Matrix de confusão:

[[1521   31]
 [ 448  113]]


In [28]:
tml.df_classifier_metrics(y_test, y_pred_lgb, y_pred_proba_lgb, ['LGBMClassifier'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
LGBMClassifier,0.773308,0.784722,0.201426,0.320567,0.829632,2113,1521,31,448,113


In [29]:
list_df = []
colunas_binarias = lt.identify_columns_binary_values(X_test)
for c in colunas_binarias:
    list_df.append(tml.df_specific_confusion_matrix(X_test, y_test, y_pred_lgb, c))
df_independent = pd.concat(list_df, axis=0)

E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes - Parte 2\TelecomX_parte2_BR\scripts\telecomx_machine_learning.py:251: RuntimeWarning: invalid value encountered in scalar divide
  precision = tp / (tp + fp)
E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes - Parte 2\TelecomX_parte2_BR\scripts\telecomx_machine_learning.py:251: RuntimeWarning: invalid value encountered in scalar divide
  precision = tp / (tp + fp)
E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes - Parte 2\TelecomX_parte2_BR\scripts\telecomx_machine_learning.py:251: RuntimeWarning: invalid value encountered in scalar divide
  precision = tp / (tp + fp)
E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes

In [30]:
df_independent.reset_index()

,index,filter_value,support,accuracy,negative predictive value,precision,recall,f1-score,TN,FP,FN,TP
0,customer_SeniorCitizen,1,349,0.690544,0.664384,0.824561,0.324138,0.465347,194,10,98,47
1,customer_Partner,1,1032,0.815891,0.821285,0.666667,0.118812,0.201681,818,12,178,24
2,customer_Dependents,1,616,0.863636,0.863262,0.888889,0.087912,0.160000,524,1,83,8
3,phone_MultipleLines,1,883,0.762174,0.762389,0.760417,0.280769,0.410112,600,23,187,73
4,account_PaperlessBilling,1,1213,0.728772,0.721402,0.790698,0.252475,0.382739,782,27,302,102
5,account_Contract_Monthly,1,1182,0.652284,0.633911,0.784722,0.229209,0.354788,658,31,380,113
6,charges_total_bin__inf_96_62_,1,242,0.681818,0.625000,0.833333,0.454545,0.588235,110,11,66,55
7,charges_total_bin__1182_80_3273_68_,1,564,0.774823,0.780443,0.636364,0.105263,0.180645,423,8,119,14
8,charges_total_bin__198_05_347_90_,1,144,0.729167,0.715385,0.857143,0.244898,0.380952,93,2,37,12
9,charges_total_bin__3273_68_4838_38_,1,214,0.864486,0.864486,NaN,0.000000,NaN,185,0,29,0


### Otimizando os hiperparâmetros com o GridSearchCV

In [32]:
param_grid = {
    'num_leaves': [15, 63],           # controle do número de folhas (mais folhas = mais complexidade)
    'max_depth': [5, 10],              # profundidade da árvore (-1 = ilimitado)
    'learning_rate': [0.01, 0.2],   # taxa de aprendizado
    'n_estimators': [100, 300, 500],     # número de árvores (boosting rounds)
    'min_child_samples': [20, 50],    # mínimo de amostras por folha
    'subsample': [0.6, 1.0],              # fração de amostras para cada árvore (bagging)
    'colsample_bytree': [0.6, 1.0],       # fração de features para cada árvore
    'reg_alpha': [0.01, 1],            # regularização L1
    'reg_lambda': [0.01, 1],           # regularização L2
}


In [33]:
# Criar modelo LightGBM
model_lgb = lgb.LGBMClassifier(
    boosting_type='gbdt',
    random_state=NUM_SEMENTE_ALEATORIA
)

In [36]:
model_grid_lgb = GridSearchCV(
    estimator=model_lgb,
    param_grid=param_grid,
    scoring='recall',
    cv=3,
    n_jobs=-1,
    verbose=1
)

In [37]:
model_grid_lgb.fit(X_train, y_train)

Fitting 3 folds for each of 768 candidates, totalling 2304 fits


GridSearchCV(cv=3, estimator=LGBMClassifier(random_state=42), n_jobs=-1,
             param_grid={'colsample_bytree': [0.6, 1.0],
                         'learning_rate': [0.01, 0.2], 'max_depth': [5, 10],
                         'min_child_samples': [20, 50],
                         'n_estimators': [100, 300, 500],
                         'num_leaves': [15, 63], 'reg_alpha': [0.01, 1],
                         'reg_lambda': [0.01, 1], 'subsample': [0.6, 1.0]},
             scoring='recall', verbose=1)

In [38]:
model_grid_lgb.best_params_

{'colsample_bytree': 0.6,
 'learning_rate': 0.2,
 'max_depth': 10,
 'min_child_samples': 50,
 'n_estimators': 300,
 'num_leaves': 63,
 'reg_alpha': 1,
 'reg_lambda': 1,
 'subsample': 0.6}

In [45]:
dict(sorted(param_grid.items()))

{'colsample_bytree': [0.6, 1.0],
 'learning_rate': [0.01, 0.2],
 'max_depth': [5, 10],
 'min_child_samples': [20, 50],
 'n_estimators': [100, 300, 500],
 'num_leaves': [15, 63],
 'reg_alpha': [0.01, 1],
 'reg_lambda': [0.01, 1],
 'subsample': [0.6, 1.0]}

In [39]:
y_pred_grid_lgb = model_grid_lgb.predict(X_test)

In [40]:
y_pred_proba_grid_lgb = model_grid_lgb.predict_proba(X_test)

In [41]:
tml.df_classifier_metrics(y_test, y_pred_grid_lgb, y_pred_proba_grid_lgb, ['LightGBM - Grid'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
LightGBM - Grid,0.793185,0.633047,0.525847,0.574489,0.830519,2113,1381,171,266,295


### Consolidação das métricas

In [49]:
df_metricas = []

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_lgb, y_pred_proba_lgb, ['LGBMClassifier']))

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_grid_lgb, y_pred_proba_grid_lgb, ['LightGBM - Grid']))

df_metricas = pd.concat(df_metricas, axis=0)
df_metricas

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
LGBMClassifier,0.773308,0.784722,0.201426,0.320567,0.829632,2113,1521,31,448,113
LightGBM - Grid,0.793185,0.633047,0.525847,0.574489,0.830519,2113,1381,171,266,295


In [38]:
df_metricas.to_clipboard()